# Exercise 04: Text Adversarial Attack on Sentiment Classifier

**Goal:** Use TextFooler to flip sentiment predictions with minimal word changes.

**Time:** ~10 min

> **Note:** Requires `textattack` and `datasets`. Install with `pip install textattack datasets`.

## Background

Text adversarial attacks must work without differentiating through discrete tokens.
**TextFooler** (Jin et al. 2020) uses a word-importance ranking to find the most influential words, then substitutes them with semantically similar synonyms (constrained by Universal Sentence Encoder similarity) until the prediction flips.

In [ ]:
# Install textattack if needed (uncomment):
# !pip install textattack datasets

import warnings, logging, os
warnings.filterwarnings("ignore")
logging.getLogger("textattack").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import nltk
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import torch
import pandas as pd

model_name = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.eval()

pipe = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

test_texts = [
    "This movie was absolutely fantastic! I loved every minute of it.",
    "Terrible film. A complete waste of time and money.",
    "A brilliant masterpiece with stunning performances throughout.",
    "Boring, predictable, and poorly acted from start to finish.",
    "One of the best cinematic experiences I have had in years!",
]

print("=== Baseline Predictions ===")
results = pipe(test_texts)
for text, res in zip(test_texts, results):
    print(f"  [{res['label']:8s} {res['score']:.2f}]  {text[:65]}...")

## 🎯 TODO: Run a TextFooler Attack

Use TextAttack to attack all 5 test examples. Follow the hints in the cell comments.

In [ ]:
# 🎯 TODO: Use TextAttack to run a TextFooler attack on the test texts above.
#
# TextFooler (Jin et al. 2020) replaces important words with semantically similar synonyms
# to flip the sentiment prediction while keeping the text readable.
#
# Steps:
#   1. Patch transformers.utils for textattack compatibility:
#        import transformers.utils as _tu
#        if not hasattr(_tu, 'is_tf_available'):
#            _tu.is_tf_available = lambda: False
#        if not hasattr(_tu, 'is_torch_available'):
#            _tu.is_torch_available = lambda: True
#
#   2. Import the required TextAttack components:
#        from textattack.models.wrappers import HuggingFaceModelWrapper
#        from textattack.attack_recipes import TextFoolerJin2019
#        from textattack import Attacker, AttackArgs
#        from textattack.datasets import Dataset as TADataset
#
#   3. Wrap the model:
#        model_wrapper = HuggingFaceModelWrapper(model, tokenizer)
#
#   4. Build the attack:
#        attack = TextFoolerJin2019.build(model_wrapper)
#
#   5. Build a dataset from test_texts.
#      TextAttack expects a list of (text, label) tuples.
#      Derive labels from the baseline predictions: POSITIVE=1, NEGATIVE=0.
#        labels_int = [1 if r['label']=='POSITIVE' else 0 for r in results]
#        dataset = TADataset(list(zip(test_texts, labels_int)))
#
#   6. Run the attack:
#        attack_args = AttackArgs(num_examples=5, disable_stdout=True, silent=True)
#        attacker = Attacker(attack, dataset, attack_args)
#        attack_results = attacker.attack_dataset()
#
#   7. Print a comparison table:
#      For each result print: original text, adversarial text, original label, new label,
#      and the percentage of words changed.
#
# HINT: Each result has:
#   res.original_result.attacked_text.text   -> original text string
#   res.perturbed_result.attacked_text.text  -> adversarial text string
#   res.original_result.output, res.perturbed_result.output  (predicted class indices)
#   type(res).__name__  -> 'SuccessfulAttackResult', 'FailedAttackResult', or 'SkippedAttackResult'

## Discussion

In [ ]:
print("""
=== Discussion ===

1. Are the adversarial examples still readable?
   TextFooler substitutes synonyms, so the text usually remains grammatical and
   semantically similar to a human reader — yet flips the model's prediction.

2. Why does synonym substitution work?
   Neural models are sensitive to specific token representations. A synonym that is
   semantically equivalent to a human may have a very different embedding, causing
   the model to cross a decision boundary.

3. How could you defend against this?
   - Adversarial training with synonym substitutions
   - Certified defenses based on interval bound propagation
   - Ensemble models trained on paraphrases
""")
